# 🤖 Detección de anomalías con Isolation Forest

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/florvela/IA-y-automatizacion-en-seguridad-defensiva/blob/main/codigos-de-ejemplo/clase_en_vivo/live_05_ml.ipynb)

**Presentar entre las diapositivas 120 y 121** (sección de detección de anomalías / UEBA).

El modelo **aprende qué es normal** y marca lo que se desvía — sin etiquetas.
Es la base de **UEBA** (comportamiento de usuarios y entidades).

<img src="https://raw.githubusercontent.com/florvela/IA-y-automatizacion-en-seguridad-defensiva/main/06-introduccion-ia-ml/images/isolation_forest.png" width="460"/>


## 1. Datos: comportamiento normal + unas anomalías
Cada evento = (logins por día, MB descargados). Casi todo es normal; inyectamos 3 raros.

In [ ]:
import numpy as np
rng = np.random.default_rng(42)

# Comportamiento NORMAL: ~3 logins/día, ~50 MB
normal = np.column_stack([rng.normal(3, 1, 200), rng.normal(50, 15, 200)])

# ANOMALÍAS: muchísimos logins y/o exfiltración de datos
anomalias = np.array([[50, 5000], [40, 8000], [2, 6000]])

X = np.vstack([normal, anomalias])
print("Total de eventos:", len(X))

## 2. Entrenar y detectar
`IsolationForest` viene en scikit-learn (ya instalado en Colab).

In [ ]:
from sklearn.ensemble import IsolationForest

modelo = IsolationForest(contamination=0.02, random_state=42)
pred = modelo.fit_predict(X)      # -1 = anomalía, 1 = normal

for i, p in enumerate(pred):
    if p == -1:
        print(f"⚠️  ANOMALÍA: {X[i][0]:.0f} logins, {X[i][1]:.0f} MB descargados")

## 3. Visualizar (opcional)

In [ ]:
import matplotlib.pyplot as plt

colores = ["red" if p == -1 else "steelblue" for p in pred]
plt.scatter(X[:, 0], X[:, 1], c=colores, alpha=0.6)
plt.xlabel("logins por día"); plt.ylabel("MB descargados")
plt.title("Rojo = anomalía detectada")
plt.show()

## Esto es UEBA
El modelo aprendió el perfil normal y devolvió una **lista corta de casos sospechosos**.
En vez de que el analista revise miles de usuarios, el sistema ya encontró los que importan.